# 0 Mount collab + Drive

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# List files in a directory
!ls '/content/drive/My Drive/Project_SS24/MedAugment'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
classification.py  evaluation.py  medaugment.py  recording		  segmentation.py
dataset		   generation.py  models.py	 requirements.txt	  selection.py
equipment.py	   __init__.py	  pipeline.py	 Segmentation_code.ipynb


# 0.25. Verify GPU

In [ ]:
import torch
print("Is GPU available:", torch.cuda.is_available())


Is GPU available: True


# 0.5. Data Management

## Image resizing to 224*224

In [ ]:
import os
from PIL import Image
import cv2

image_folder = '/content/drive/My Drive/Project_SS24/MedAugment/dataset/segmentation/cvc/baseline/training'

# Create a new folder for the resized images if it doesn't exist
resized_folder = '/content/drive/My Drive/Project_SS24/MedAugment/dataset/segmentation/cvc/baseline/training'
if not os.path.exists(resized_folder):
    os.makedirs(resized_folder)

# Iterate over each file in the folder
for filename in os.listdir(image_folder):
    if filename.endswith(".jpg") or filename.endswith(".png"):  # You can add more image extensions if needed
        image_path = os.path.join(image_folder, filename)
        img = Image.open(image_path)

        # Resize the image to 224x224
        img_resized = img.resize((224, 224))

        # Save the resized image to the new folder
        img_resized.save(os.path.join(resized_folder, filename))


## Add _mask to the end of masks' names

In [ ]:
import os

# Define the path to your images directory
images_dir = '/dataset/segmentation/cvc/baseline/test_mask'  # Replace with the path to your images folder

# Loop through all files in the directory
for filename in os.listdir(images_dir):
    # Check if the file is an image (assuming .png format)
    if filename.endswith('.png'):
        # Define the new filename with '_mask' suffix
        new_filename = filename.replace('.png', '_mask.png')

        # Define the full paths for the old and new filenames
        old_file = os.path.join(images_dir, filename)
        new_file = os.path.join(images_dir, new_filename)

        # Rename the file
        os.rename(old_file, new_file)

        print(f'Renamed: {filename} -> {new_filename}')


# 1. Run MedAugment

## Make main() to simulate the arguments

In [ ]:
import albumentations as A
import torch
import math
import random
import os
import cv2
import shutil
import numpy as np
from torchvision import transforms
from PIL import Image

def odd_conversion(num):
    num = math.ceil(num)
    if num % 2 == 0:
        num += 1
    return num

def med_augment(data_path, name, level, number_branch, mask_i=False, shield=False):
    if mask_i:
        image_path = f"{data_path}{name}"
        mask_path = f"{image_path}_mask"
        output_path = f"{os.path.dirname(os.path.dirname(data_path))}/medaugment/{name}/"
        out_mask = f"{os.path.dirname(os.path.dirname(data_path))}/medaugment/{name}_mask/"
    else:
        image_path = data_path + name
        output_path = f"{os.path.dirname(os.path.dirname(os.path.dirname(data_path)))}/medaugment/training/{name}/"

    transform = A.Compose([
        A.ColorJitter(brightness=0.04 * level, contrast=0, saturation=0, hue=0, p=0.2 * level),
        A.ColorJitter(brightness=0, contrast=0.04 * level, saturation=0, hue=0, p=0.2 * level),
        A.Posterize(num_bits=math.floor(8 - 0.8 * level), p=0.2 * level),
        A.Sharpen(alpha=(0.04 * level, 0.1 * level), lightness=(1, 1), p=0.2 * level),
        A.GaussianBlur(blur_limit=(3, odd_conversion(3 + 0.8 * level)), p=0.2 * level),
        A.GaussNoise(var_limit=(2 * level, 10 * level), mean=0, per_channel=True, p=0.2 * level),
        A.Rotate(limit=4 * level, interpolation=1, border_mode=0, value=0, mask_value=None, rotate_method='largest_box',
                 crop_border=False, p=0.2 * level),
        A.HorizontalFlip(p=0.2 * level),
        A.VerticalFlip(p=0.2 * level),
        A.Affine(scale=(1 - 0.04 * level, 1 + 0.04 * level), translate_percent=None, translate_px=None, rotate=None,
                 shear=None, interpolation=1, mask_interpolation=0, cval=0, cval_mask=0, mode=0, fit_output=False,
                 keep_ratio=True, p=0.2 * level),
        A.Affine(scale=None, translate_percent=None, translate_px=None, rotate=None,
                 shear={'x': (0, 2 * level), 'y': (0, 0)}
                 , interpolation=1, mask_interpolation=0, cval=0, cval_mask=0, mode=0, fit_output=False,
                 keep_ratio=True, p=0.2 * level),  # x
        A.Affine(scale=None, translate_percent=None, translate_px=None, rotate=None,
                 shear={'x': (0, 0), 'y': (0, 2 * level)}
                 , interpolation=1, mask_interpolation=0, cval=0, cval_mask=0, mode=0, fit_output=False,
                 keep_ratio=True, p=0.2 * level),
        A.Affine(scale=None, translate_percent={'x': (0, 0.02 * level), 'y': (0, 0)}, translate_px=None, rotate=None,
                 shear=None, interpolation=1, mask_interpolation=0, cval=0, cval_mask=0, mode=0, fit_output=False,
                 keep_ratio=True, p=0.2 * level),
        A.Affine(scale=None, translate_percent={'x': (0, 0), 'y': (0, 0.02 * level)}, translate_px=None, rotate=None,
                 shear=None, interpolation=1, mask_interpolation=0, cval=0, cval_mask=0, mode=0, fit_output=False,
                 keep_ratio=True, p=0.2 * level)
    ])

    for j, file_name in enumerate(os.listdir(image_path)):
        if file_name.endswith(".png") or file_name.endswith(".jpg"):
            file_path = os.path.join(image_path, file_name)
            file_n, file_s = file_name.split(".")[0], file_name.split(".")[1]
            image = cv2.imread(file_path)
            if mask_i: mask = cv2.imread(f"{mask_path}/{file_n}_mask.{file_s}")
            strategy = [(1, 2), (0, 3), (0, 2), (1, 1)]
            for i in range(number_branch):
                if number_branch != 4:
                    employ = random.choice(strategy)
                else:
                    index = random.randrange(len(strategy))
                    employ = strategy.pop(index)
                level, shape = random.sample(transform[:6], employ[0]), random.sample(transform[6:], employ[1])
                img_transform = A.Compose([*level, *shape])
                random.shuffle(img_transform.transforms)
                if not os.path.exists(output_path): os.makedirs(output_path)
                if mask_i:
                    transformed = img_transform(image=image, mask=mask)
                    transformed_image, transformed_mask = transformed['image'], transformed['mask']
                    cv2.imwrite(f"{output_path}/{file_n}_{i+1}.{file_s}", transformed_image)
                    cv2.imwrite(f"{out_mask}/{file_n}_{i+1}_mask.{file_s}", transformed_mask)
                else:
                    transformed = img_transform(image=image)
                    transformed_image = transformed['image']
                    cv2.imwrite(f"{output_path}/{file_n}_{i+1}.{file_s}", transformed_image)
                if not shield:
                    cv2.imwrite(f"{output_path}/{file_n}_{number_branch+1}.{file_s}", image)
                    if mask_i: cv2.imwrite(f"{out_mask}/{file_n}_{number_branch+1}_mask.{file_s}", mask)

def generate_datasets(train_type, dataset, seed, level, number_branch):
    torch.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed(seed)

    if train_type == "classification":
        print('Executing data augmentation for image classification...')
        data_path = f"/content/drive/MyDrive/Project_SS24/MedAugment/dataset/classification/{dataset}/baseline/training/"
        folder_path = f"/content/drive/MyDrive/Project_SS24/MedAugment/dataset/classification/{dataset}/"
        n = len([name for name in os.listdir(f"{folder_path}/baseline/training") if
                 os.path.isdir(os.path.join(f"{folder_path}/baseline/training", name))])

        for folder in ["medaugment"]:
            shutil.copytree(f"{folder_path}baseline", f"{folder_path}{folder}", ignore=shutil.ignore_patterns("training"))
            training_folder_path = f"{folder_path}{folder}/training"
            os.makedirs(training_folder_path)
            for i in range(n):
                os.makedirs(f"{training_folder_path}/n{i}")

        for i in range(n):
            name = f"n{i}"
            med_augment(data_path, name, level, number_branch)
    else:
        print('Executing data augmentation for image segmentation...')
        data_path = f"/content/drive/MyDrive/Project_SS24/MedAugment/dataset/segmentation/{dataset}/baseline/"
        folder_path = f"/content/drive/MyDrive/Project_SS24/MedAugment/dataset/segmentation/{dataset}/"

        for folder in ["medaugment"]:
            shutil.copytree(f"{folder_path}baseline", f"{folder_path}{folder}", ignore=shutil.ignore_patterns("training", "training_mask"))
            os.makedirs(f"{folder_path}{folder}/training")
            os.makedirs(f"{folder_path}{folder}/training_mask")

        folder_list = ["training"]
        for i in range(len(folder_list)):
            name = folder_list[i]
            med_augment(data_path, name, level, number_branch, mask_i=True)

def main():
    # Manually define the arguments
    args = {
        'dataset': 'cvc',  # Replace with your actual dataset name
        'train_type': 'segmentation',  # Or 'segmentation'
        'level': 5,                      # Augmentation level
        'number_branch': 4,              # Number of branches
        'seed': 8                        # Seed value
    }
    generate_datasets(**args)

if __name__ == '__main__':
    main()


Executing data augmentation for image segmentation...


/usr/local/lib/python3.10/dist-packages/pydantic/main.py:364: UserWarning: Pydantic serializer warnings:
  Expected `Union[float, tuple[float, float]]` but got `list` - serialized value may not be as expected
  Expected `Union[float, tuple[float, float]]` but got `list` - serialized value may not be as expected
  Expected `Union[float, tuple[float, float]]` but got `list` - serialized value may not be as expected
  Expected `Union[float, tuple[float, float]]` but got `list` - serialized value may not be as expected
  return self.__pydantic_serializer__.to_python(


# 1.5. Get ready before segmentation

## equipment.py

In [ ]:
import torch


def equipment():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if torch.cuda.is_available():
        torch.set_default_tensor_type(torch.cuda.FloatTensor)
    else:
        print("Warning: cuda unavailable, switch to %s instead" % device)

    return device


## evaluation.py

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
#import SimpleITK as sitk
from sklearn.metrics import confusion_matrix
import torch
import numpy as np
import torch.nn as nn
import json


def format_conversion(tl, pl):
    tru, pre = [], []
    for i in range(len(tl)):
        for j in range(len(tl[i])):
            pre.append(int(pl[i][j]))
            tru.append(int(tl[i][j]))

    return tru, pre


def confusion(con_str, num_class, tl, pl, save=False):
    tru, pre = format_conversion(tl, pl)
    conf_mat = confusion_matrix(tru, pre)  # (bs, num_cla)
    te_acc, tev_npv, te_ppv, te_sen, te_spe, te_fos = calculate_metrics_cls(conf_mat, con_str)
    if save:
        labels = [f"Class {i+1}" for i in range(num_class)]
        df_cm = pd.DataFrame(conf_mat, index=labels, columns=labels)
        sns.set(font_scale=1.4)
        heatmap = sns.heatmap(df_cm, annot=True, fmt="d", cmap="YlGnBu", square=False)
        heatmap.yaxis.set_ticklabels(heatmap.yaxis.get_ticklabels(), rotation=90, ha='right')
        heatmap.xaxis.set_ticklabels(heatmap.xaxis.get_ticklabels(), rotation=0, ha='center')
        matplotlib.rcParams['font.sans-serif'] = 'Arial'
        plt.ylabel('True')
        plt.xlabel('Predicted')
        plt.savefig('/content/drive/MyDrive/Project_SS24/MedAugment/recording/classification/' + str(con_str) + 'matrix.pdf', bbox_inches='tight', pad_inches=0.0, dpi=300)
        plt.show()

    return te_acc, tev_npv, te_ppv, te_sen, te_spe, te_fos


def calculate_metrics_cls(conf_mat, con_str, e=1e-6):
    num_class = conf_mat.shape[0]
    TP = np.diag(conf_mat)
    FN = conf_mat.sum(axis=1) - TP
    FP = conf_mat.sum(axis=0) - TP
    TN = conf_mat.sum() - (TP + FN + FP)

    accuracy = np.sum(TP) / np.sum(conf_mat)
    npv = TN / (TN + FN + e)
    ppv = TP / (TP + FP + e)
    sensitivity = TP / (TP + FN + e)
    specificity = TN / (TN + FP + e)
    f1_score = 2 * ppv * sensitivity / (ppv + sensitivity + e)

    macro_npv = np.mean(npv)
    macro_ppv = np.mean(ppv)
    macro_sensitivity = np.mean(sensitivity)
    macro_specificity = np.mean(specificity)
    macro_f1_score = np.mean(f1_score)

    metrics = {
        "accuracy": accuracy,
        "npv": {f"Class {i + 1}": npv[i] for i in range(num_class)},
        "ppv": {f"Class {i + 1}": ppv[i] for i in range(num_class)},
        "sensitivity": {f"Class {i + 1}": sensitivity[i] for i in range(num_class)},
        "specificity": {f"Class {i + 1}": specificity[i] for i in range(num_class)},
        "f1_score": {f"Class {i + 1}": f1_score[i] for i in range(num_class)},
        "macro_npv": macro_npv,
        "macro_ppv": macro_ppv,
        "macro_sensitivity": macro_sensitivity,
        "macro_specificity": macro_specificity,
        "macro_f1_score": macro_f1_score,
    }

    with open('/content/drive/MyDrive/Project_SS24/MedAugment/recording/classification/' + str(con_str) + 'metrics.json', "w") as f:
        json.dump(metrics, f, indent=4)

    return accuracy, macro_npv, macro_ppv, macro_sensitivity, macro_specificity, macro_f1_score


def soft_iou(out, target, e=1e-6):
    target, out = target.squeeze(dim=1), out.squeeze(dim=1)
    target, out = target.reshape(target.shape[0], -1), out.reshape(target.shape[0], -1)
    out = torch.sigmoid(out)
    num = (out*target).sum(1, True)
    den = (out+target-out*target).sum(1, True) + e
    iou = num / den
    cost = 1 - iou

    return cost.squeeze()


class SoftIoULoss(nn.Module):
    def __init__(self):
        super(SoftIoULoss,self).__init__()
    def forward(self, y_pred, y_true):
        costs = soft_iou(y_pred, y_true).view(-1,1)
        costs = torch.mean(costs)
        return costs


def calculate_metrics_seg(target, out, e=1e-6):
    out = torch.sigmoid(out)
    out_binary = (out > 0.5).int()
    target_binary = (target > 0.5).int()
    intersection = torch.logical_and(out_binary, target_binary).sum(dim=(2, 3))
    union_dice = out_binary.sum(dim=(2, 3)) + target_binary.sum(dim=(2, 3))
    union_iou = union_dice - intersection
    dice_score = 2 * intersection / (union_dice + e)
    iou_score = intersection / (union_iou + e)
    correct_pixels = torch.eq(out_binary, target_binary).sum()
    total_pixels = target_binary.numel()
    pixel_accuracy = correct_pixels / total_pixels

    return dice_score.mean(), iou_score.mean(), pixel_accuracy


# models.py

In [ ]:
!pip install segmentation_models_pytorch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 5.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.5/68.5 kB 6.9 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  U

In [ ]:
import torch
from torchvision.models.feature_extraction import create_feature_extractor
from torchvision import models
import segmentation_models_pytorch as smp


class VGGNet(torch.nn.Module):
    def __init__(self, num_class):
        super(VGGNet, self).__init__()
        self.model = models.vgg16(weights='VGG16_Weights.IMAGENET1K_V1')
        self.model.classifier = torch.nn.Sequential(
            torch.nn.Linear(25088, 4096),
            torch.nn.ReLU(inplace=True),
            torch.nn.Dropout(p=0.5, inplace=False),
            torch.nn.Linear(4096, 4096),
            torch.nn.ReLU(inplace=True),
            torch.nn.Dropout(p=0.5, inplace=False),
            torch.nn.Linear(4096, num_class),
        )

    def forward(self, x):
        for name, param in self.model.named_parameters():
            if 'classifier' not in name:
                param.requires_grad = False
        return self.model(x)


class ResNeXt(torch.nn.Module):
    def __init__(self, num_class):
        super(ResNeXt, self).__init__()
        self.model = models.resnext50_32x4d(weights='ResNeXt50_32X4D_Weights.IMAGENET1K_V1')
        self.model.fc = torch.nn.Linear(2048, num_class)

    def forward(self, x):
        for name, param in self.model.named_parameters():
            if 'fc' not in name:
                param.requires_grad = False
        return self.model(x)


class ConvNeXt(torch.nn.Module):
    def __init__(self, num_class):
        super(ConvNeXt, self).__init__()
        self.model = models.convnext_tiny(weights='ConvNeXt_Tiny_Weights.IMAGENET1K_V1')
        self.model.classifier = torch.nn.Sequential(
            models.convnext.LayerNorm2d(768, eps=1e-06),
            torch.nn.Flatten(),
            torch.nn.Linear(768, num_class)
        )

    def forward(self, x):
        for name, param in self.model.named_parameters():
            if 'classifier' not in name:
                param.requires_grad = False
        return self.model(x)


class UNetPP(torch.nn.Module):
    def __init__(self):
        super(UNetPP, self).__init__()
        self.model = smp.UnetPlusPlus(
            encoder_name="resnet18",
            encoder_weights="imagenet",
            in_channels=3,
            classes=1,
        )

    def forward(self, x):
        return self.model(x)


class FPN(torch.nn.Module):
    def __init__(self):
        super(FPN, self).__init__()
        self.model = smp.FPN(
            encoder_name="resnet18",
            encoder_weights="imagenet",
            in_channels=3,
            classes=1,
        )

    def forward(self, x):
        return self.model(x)


class DeepLab(torch.nn.Module):
    def __init__(self):
        super(DeepLab, self).__init__()
        self.model = smp.DeepLabV3(
            encoder_name="resnet18",
            encoder_weights="imagenet",
            in_channels=3,
            classes=1,
        )

    def forward(self, x):
        return self.model(x)


# pipeline.py

In [ ]:
import os
from PIL import Image
import torch.utils.data as tud
import torch
from torchvision import transforms
from torchvision.datasets import ImageFolder


def cls_data_aug(dataset, index, bs, device):
    sets = ["medaugment"]
    set_type = sets[index-1]
    transform = transforms.ToTensor()

    train_d = ImageFolder(f"/content/drive/MyDrive/Project_SS24/MedAugment/dataset/classification/{dataset}/{set_type}/training", transform=transform)
    val_d = ImageFolder(f"/content/drive/MyDrive/Project_SS24/MedAugment/dataset/classification/{dataset}/{set_type}/validation", transform=transform)
    test_d = ImageFolder(f"/content/drive/MyDrive/Project_SS24/MedAugment/dataset/classification/{dataset}/{set_type}/test", transform=transform)

    if str(device) == 'cuda':
        train_l = tud.DataLoader(train_d, batch_size=bs, shuffle=True, generator=torch.Generator(device='cuda'))
        val_l = tud.DataLoader(val_d, batch_size=bs, shuffle=True, generator=torch.Generator(device='cuda'))
        test_l = tud.DataLoader(test_d, batch_size=bs, shuffle=True, generator=torch.Generator(device='cuda'))
    else:
        train_l = tud.DataLoader(train_d, batch_size=bs, shuffle=True)
        val_l = tud.DataLoader(val_d, batch_size=bs, shuffle=True)
        test_l = tud.DataLoader(test_d, batch_size=bs, shuffle=True)

    return train_l, val_l, test_l, train_d, val_d, test_d


class SegDataset(tud.Dataset):
    def __init__(self, path, transform):
        super(SegDataset, self).__init__()
        self.img_path = path
        self.mask_path = path + "_mask"
        self.transform = transform
        self.img_name = [i for i in os.listdir(self.img_path) if i.endswith(".png") or i.endswith(".jpg")]

    def __getitem__(self, item):
        img = Image.open(os.path.join(self.img_path, self.img_name[item]))
        mask_file = os.path.join(self.mask_path, self.img_name[item].split(".")[0] + "_mask")
        if os.path.exists(mask_file + ".png"):
            mask = Image.open(mask_file + ".png")
        elif os.path.exists(mask_file + ".jpg"):
            mask = Image.open(mask_file + ".jpg")

        if not self.transform is None:
            img_rgb, mask_rgb = img.convert('RGB'), mask.convert('RGB')
            img, mask = self.transform(img_rgb), self.transform(mask_rgb)

        else:
            img, mask = img, mask

        return self.img_name[item], img, mask[0,:,:].unsqueeze(0)

    def __len__(self):
        return len(self.img_name)


def seg_data_aug(dataset, index, bs, device):
    sets = ["medaugment"]
    set_type = sets[index-1]
    transform = transforms.ToTensor()

    train_d = SegDataset(f"/content/drive/MyDrive/Project_SS24/MedAugment/dataset/segmentation/{dataset}/{set_type}/training", transform=transform)
    val_d = SegDataset(f"/content/drive/MyDrive/Project_SS24/MedAugment/dataset/segmentation/{dataset}/{set_type}/validation", transform=transform)
    test_d = SegDataset(f"/content/drive/MyDrive/Project_SS24/MedAugment/dataset/segmentation/{dataset}/{set_type}/test", transform=transform)

    if str(device) == 'cuda':
        train_l = tud.DataLoader(train_d, batch_size=bs, shuffle=True, generator=torch.Generator(device='cuda'))
        val_l = tud.DataLoader(val_d, batch_size=bs, shuffle=True, generator=torch.Generator(device='cuda'))
        test_l = tud.DataLoader(test_d, batch_size=bs, shuffle=True, generator=torch.Generator(device='cuda'))
    else:
        train_l = tud.DataLoader(train_d, batch_size=bs, shuffle=True)
        val_l = tud.DataLoader(val_d, batch_size=bs, shuffle=True)
        test_l = tud.DataLoader(test_d, batch_size=bs, shuffle=True)

    return train_l, val_l, test_l, train_d, val_d, test_d


# selection.py

In [ ]:
def cls_model_sel(model, num_class):
    model_type = {
        'VGGNet': VGGNet,
        'ResNeXt': ResNeXt,
        'ConvNeXt': ConvNeXt,
    }
    constructor = model_type.get(model)
    model = constructor(num_class)

    return model


def seg_model_sel(model):
    model_type = {
        'UNetPP': UNetPP,
        'FPN': FPN,
        'DeepLab': DeepLab,
    }
    constructor = model_type.get(model)
    model = constructor()

    return model


# 2. Run segmentation

In [ ]:
def seg_model_sel(model_name, device):
    if model_name == 'UNetPP':
        model = smp.UnetPlusPlus(
            encoder_name="resnet18",        # choose encoder, e.g. mobilenet_v2 or efficientnet-b7
            encoder_weights="imagenet",     # use `imagenet` pre-trained weights for encoder initialization
            in_channels=3,                  # model input channels (1 for grayscale, 3 for RGB)
            classes=1,                      # model output channels (number of classes in your dataset)
        )
    elif model_name == 'FPN':
        model = smp.FPN(
            encoder_name="resnet34",        # choose encoder
            encoder_weights="imagenet",     # use pre-trained weights
            in_channels=3,                  # model input channels
            classes=1,                      # model output channels
        )
    elif model_name == 'DeepLabP':
        model = smp.DeepLabV3Plus(
            encoder_name="resnet34",        # choose encoder
            encoder_weights="imagenet",     # use pre-trained weights
            in_channels=3,                  # model input channels
            classes=1,                      # model output channels
        )
    else:
        raise ValueError(f"Model {model_name} not supported")

    # Move model to the appropriate device
    model.to(device)
    return model


In [ ]:
import time
import pandas as pd
import torch
import numpy as np
import os
import torch.optim.lr_scheduler as lr_sche
#from utils import equipment, seg_model_sel, SoftIoULoss, seg_data_aug, calculate_metrics_seg
from torch.optim import Adam
import segmentation_models_pytorch as smp
import cv2
import random

def seed_torch(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

# --------------------------------------------------------Train function---------------------------------------------------------------------

def train(model, dataset, train_type, index, seed, num_class, decay, bs, value, epoch, lr, min_loss):
    # Set the random seed for reproducibility
    seed_torch(seed)

    # Select the device (GPU if available, otherwise CPU)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Move the model to the selected device
    model = seg_model_sel(model, device)

    # Define the loss function and move it to the device if necessary
    loss_func = SoftIoULoss().to(device)

    # Define the optimizer
    optimizer = Adam(model.parameters(), lr=lr, weight_decay=value) if decay else Adam(model.parameters(), lr=lr)

    # Define the learning rate scheduler
    scheduler = lr_sche.StepLR(optimizer, step_size=20, gamma=0.9)

    # Get the data loaders and move data to the device in the loop
    train_l, val_l, test_l, train_d, val_d, test_d = seg_data_aug(dataset, index, bs, device)

    con_str = f"{model.__class__.__name__}-{dataset}-{decay}-{index}-{seed}-"
    stop_index = 0

    print('Batch size: %d\nLearning rate: %s\nNumber of epoch: %d' % (bs, lr, epoch),
          file=open(f"/content/drive/MyDrive/Project_SS24/MedAugment/recording/{train_type}/{con_str}log.txt", "w"))

    for i in range(epoch):
        # Initialize metrics for each epoch
        t_loss_b, v_loss_b, t_dice_b, v_dice_b, v_iou_b, v_pa_b = 0, 0, 0, 0, 0, 0

        print('\nEpoch %d/%d \n' % (i + 1, epoch) + '-' * 60,
              file=open(f"/content/drive/MyDrive/Project_SS24/MedAugment/recording/{train_type}/{con_str}log.txt", "a"))

        since = time.time()
        model.train()

        # Training loop
        for step, (name, t_x, t_y) in enumerate(train_l):
            t_x, t_y = t_x.to(device), t_y.to(device)  # Move inputs and labels to device
            output = model(t_x)
            loss = loss_func(output, t_y)
            dice, _, _ = calculate_metrics_seg(output, t_y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            t_loss_b += loss.item() * t_x.size(0)
            t_dice_b += dice.item() * t_x.size(0)

        model.eval()

        # Validation loop
        for step, (name, v_x, v_y) in enumerate(val_l):
            v_x, v_y = v_x.to(device), v_y.to(device)  # Move inputs and labels to device
            output = model(v_x)
            loss = loss_func(output, v_y)
            dice, iou, pa = calculate_metrics_seg(output, v_y)

            v_loss_b += loss.item() * v_x.size(0)
            v_dice_b += dice.item() * v_x.size(0)
            v_iou_b += iou.item() * v_x.size(0)
            v_pa_b += pa.item() * v_x.size(0)

        t_c = time.time() - since
        t_loss, t_dice = t_loss_b / len(train_d.img_name), t_dice_b / len(train_d.img_name)
        v_loss, v_dice, v_iou, v_pa = v_loss_b / len(val_d.img_name), v_dice_b / len(val_d.img_name), v_iou_b / len(val_d.img_name), v_pa_b / len(val_d.img_name)

        scheduler.step()
        print('Train and validation done in %d m %d s \nTrain loss: %.3f, dice: %.3f; Val loss: %.3f, dice: %.3f, iou: %.3f, pa: %.3f' % (
        t_c // 60, t_c % 60, t_loss, t_dice, v_loss, v_dice, v_iou, v_pa),
        file=open(f"/content/drive/MyDrive/Project_SS24/MedAugment/recording/{train_type}/{con_str}log.txt", "a"))

        # Test loop
        te_loss_b, te_dice_b, te_iou_b, te_pa_b = 0, 0, 0, 0
        since = time.time()
        model.eval()
        for step, (name, te_x, te_y) in enumerate(test_l):
            te_x, te_y = te_x.to(device), te_y.to(device)  # Move inputs and labels to device
            output = model(te_x)
            loss = loss_func(output, te_y)
            dice, iou, pa = calculate_metrics_seg(output, te_y)

            te_loss_b += loss.item() * te_x.size(0)
            te_dice_b += dice.item() * te_x.size(0)
            te_iou_b += iou.item() * te_x.size(0)
            te_pa_b += pa.item() * te_x.size(0)

            if v_loss < min_loss:
                for i in range(output.size(0)):
                    pre_path = f"./recording/{train_type}/{con_str}pre"
                    os.makedirs(pre_path, exist_ok=True)
                    img_name, img_ext = os.path.splitext(name[i])
                    img = torch.sigmoid(output[i])
                    img = (img > 0.5).float() * 255
                    img = img.detach().squeeze().cpu().numpy()
                    img_path = os.path.join(pre_path, img_name + img_ext)
                    cv2.imwrite(img_path, img)

        t_c = time.time() - since
        te_loss, te_dice, te_iou, te_pa = te_loss_b / len(test_d.img_name), te_dice_b / len(test_d.img_name), te_iou_b / len(test_d.img_name), te_pa_b / len(test_d.img_name)
        print('Test done in %d m %d s \nTest loss: %.3f, dice: %.3f, iou: %.3f, pa: %.3f' % (t_c // 60, t_c % 60,
              te_loss, te_dice, te_iou, te_pa),
              file=open(f"/content/drive/MyDrive/Project_SS24/MedAugment/recording/{train_type}/{con_str}log.txt", "a"))

        if v_loss < min_loss:
            stop_index = 0
            min_loss = v_loss
            torch.save(model, f"/content/drive/MyDrive/Project_SS24/MedAugment/recording/{train_type}/{con_str}model.pkl")
            print("Model saved", file=open(f"/content/drive/MyDrive/Project_SS24/MedAugment/recording/{train_type}/{con_str}log.txt", "a"))
        else:
            stop_index += 1

        if stop_index == 8:
            print("Early stopping triggered", file=open(f"/content/drive/MyDrive/Project_SS24/MedAugment/recording/{train_type}/{con_str}log.txt", "a"))
            break
# -----------------------------------------------------------------------------------------------------------------------------

def segmentation():
    # Manually define the arguments instead of using argparse
    args = {
        'model': 'UNetPP',           # Choose your model: 'UNetPP', 'FPN', or 'DeepLabP'
        'dataset': 'cvc',          # Choose your dataset: 'lung', 'kvasir', 'cvc', or 'covid'
        'train_type': 'segmentation',
        'index': 1,                  # Index for method of run
        'seed': 1,                   # Random seed
        'num_class': 1,              # Number of classes
        'decay': True,               # Setting of weight decay
        'bs': 128,                   # Batch size for training
        'value': 1e-2,               # Decay value
        'epoch': 40,                 # Number of epochs
        'lr': 0.002,                 # Learning rate
        'min_loss': 1e4              # Minimum loss
    }
    train(**args)

if __name__ == '__main__':
    segmentation()


RuntimeError: Expected a 'cpu' device type for generator but found 'cuda'

## ChatGPT code:

In [ ]:
import time
import pandas as pd
import torch
import numpy as np
import os
import torch.optim.lr_scheduler as lr_sche
from torch.optim import Adam
import segmentation_models_pytorch as smp
import cv2
import random

def seed_torch(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

# --------------------------------------------------------Train function---------------------------------------------------------------------

def seg_model_sel(model_name, device):
    if model_name == 'UNetPP':
        model = smp.UnetPlusPlus(
            encoder_name="resnet18",        # choose encoder, e.g. mobilenet_v2 or efficientnet-b7
            encoder_weights="imagenet",     # use `imagenet` pre-trained weights for encoder initialization
            in_channels=3,                  # model input channels (1 for grayscale, 3 for RGB)
            classes=1,                      # model output channels (number of classes in your dataset)
        )
    elif model_name == 'FPN':
        model = smp.FPN(
            encoder_name="resnet34",        # choose encoder
            encoder_weights="imagenet",     # use pre-trained weights
            in_channels=3,                  # model input channels
            classes=1,                      # model output channels
        )
    elif model_name == 'DeepLabP':
        model = smp.DeepLabV3Plus(
            encoder_name="resnet34",        # choose encoder
            encoder_weights="imagenet",     # use pre-trained weights
            in_channels=3,                  # model input channels
            classes=1,                      # model output channels
        )
    else:
        raise ValueError(f"Model {model_name} not supported")

    # Move model to the GPU
    model.to(device)
    return model

def train(model, dataset, train_type, index, seed, num_class, decay, bs, value, epoch, lr, min_loss):
    # Set the random seed for reproducibility
    seed_torch(seed)

    # Select the device (GPU if available, otherwise CPU)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Move the model to the selected device
    model = seg_model_sel(model, device)

    # Define the loss function and move it to the device if necessary
    loss_func = SoftIoULoss().to(device)

    # Define the optimizer
    optimizer = Adam(model.parameters(), lr=lr, weight_decay=value) if decay else Adam(model.parameters(), lr=lr)

    # Define the learning rate scheduler
    scheduler = lr_sche.StepLR(optimizer, step_size=20, gamma=0.9)

    # Get the data loaders and move data to the device in the loop
    train_l, val_l, test_l, train_d, val_d, test_d = seg_data_aug(dataset, index, bs, device)

    con_str = f"{model.__class__.__name__}-{dataset}-{decay}-{index}-{seed}-"
    stop_index = 0

    print('Batch size: %d\nLearning rate: %s\nNumber of epoch: %d' % (bs, lr, epoch),
          file=open(f"/content/drive/MyDrive/Project_SS24/MedAugment/recording/{train_type}/{con_str}log.txt", "w"))

    for i in range(epoch):
        # Initialize metrics for each epoch
        t_loss_b, v_loss_b, t_dice_b, v_dice_b, v_iou_b, v_pa_b = 0, 0, 0, 0, 0, 0

        print('\nEpoch %d/%d \n' % (i + 1, epoch) + '-' * 60,
              file=open(f"/content/drive/MyDrive/Project_SS24/MedAugment/recording/{train_type}/{con_str}log.txt", "a"))

        since = time.time()
        model.train()

        # Training loop
        for step, (name, t_x, t_y) in enumerate(train_l):
            t_x, t_y = t_x.to(device), t_y.to(device)  # Move inputs and labels to device
            output = model(t_x)
            loss = loss_func(output, t_y)
            dice, _, _ = calculate_metrics_seg(output, t_y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            t_loss_b += loss.item() * t_x.size(0)
            t_dice_b += dice.item() * t_x.size(0)

        model.eval()

        # Validation loop
        for step, (name, v_x, v_y) in enumerate(val_l):
            v_x, v_y = v_x.to(device), v_y.to(device)  # Move inputs and labels to device
            output = model(v_x)
            loss = loss_func(output, v_y)
            dice, iou, pa = calculate_metrics_seg(output, v_y)

            v_loss_b += loss.item() * v_x.size(0)
            v_dice_b += dice.item() * v_x.size(0)
            v_iou_b += iou.item() * v_x.size(0)
            v_pa_b += pa.item() * v_x.size(0)

        t_c = time.time() - since
        t_loss, t_dice = t_loss_b / len(train_d.img_name), t_dice_b / len(train_d.img_name)
        v_loss, v_dice, v_iou, v_pa = v_loss_b / len(val_d.img_name), v_dice_b / len(val_d.img_name), v_iou_b / len(val_d.img_name), v_pa_b / len(val_d.img_name)

        scheduler.step()
        print('Train and validation done in %d m %d s \nTrain loss: %.3f, dice: %.3f; Val loss: %.3f, dice: %.3f, iou: %.3f, pa: %.3f' % (
        t_c // 60, t_c % 60, t_loss, t_dice, v_loss, v_dice, v_iou, v_pa),
        file=open(f"/content/drive/MyDrive/Project_SS24/MedAugment/recording/{train_type}/{con_str}log.txt", "a"))

        # Test loop
        te_loss_b, te_dice_b, te_iou_b, te_pa_b = 0, 0, 0, 0
        since = time.time()
        model.eval()
        for step, (name, te_x, te_y) in enumerate(test_l):
            te_x, te_y = te_x.to(device), te_y.to(device)  # Move inputs and labels to device
            output = model(te_x)
            loss = loss_func(output, te_y)
            dice, iou, pa = calculate_metrics_seg(output, te_y)

            te_loss_b += loss.item() * te_x.size(0)
            te_dice_b += dice.item() * te_x.size(0)
            te_iou_b += iou.item() * te_x.size(0)
            te_pa_b += pa.item() * te_x.size(0)

            if v_loss < min_loss:
                for i in range(output.size(0)):
                    pre_path = f"./recording/{train_type}/{con_str}pre"
                    os.makedirs(pre_path, exist_ok=True)
                    img_name, img_ext = os.path.splitext(name[i])
                    img = torch.sigmoid(output[i])
                    img = (img > 0.5).float() * 255
                    img = img.detach().squeeze().cpu().numpy()
                    img_path = os.path.join(pre_path, img_name + img_ext)
                    cv2.imwrite(img_path, img)

        t_c = time.time() - since
        te_loss, te_dice, te_iou, te_pa = te_loss_b / len(test_d.img_name), te_dice_b / len(test_d.img_name), te_iou_b / len(test_d.img_name), te



def segmentation():
    # Manually define the arguments instead of using argparse
    args = {
        'model': 'UNetPP',           # Choose your model: 'UNetPP', 'FPN', or 'DeepLabP'
        'dataset': 'cvc',          # Choose your dataset: 'lung', 'kvasir', 'cvc', or 'covid'
        'train_type': 'segmentation',
        'index': 1,                  # Index for method of run
        'seed': 1,                   # Random seed
        'num_class': 1,              # Number of classes
        'decay': True,               # Setting of weight decay
        'bs': 128,                   # Batch size for training
        'value': 1e-2,               # Decay value
        'epoch': 40,                 # Number of epochs
        'lr': 0.002,                 # Learning rate
        'min_loss': 1e4              # Minimum loss
    }
    train(**args)

if __name__ == '__main__':
    segmentation()

RuntimeError: Expected a 'cpu' device type for generator but found 'cuda'

# Random Cropping

Write a code to do the following task:
1. random resizing and cropping:
• Random Resizing: Adjusting the size of an image to a randomly selected scale within a
specified range, potentially altering its aspect ratio.
• Random Cropping: Extracting a random rectangular portion of the resized image, which
may include or exclude important features.
2. label-aware resizing and cropping:
• Label-Aware Resizing: Resizing the image while ensuring that regions of interest (labeled
areas) are preserved and remain within the frame.
• Label-Aware Cropping: Cropping the image in a way that always includes the labeled
regions, ensuring that critical areas are not excluded.

In [ ]:
# prompt: Write a code to do the following task:
# random resizing and cropping: • Random Resizing: Adjusting the size of an image to a randomly selected scale within a specified range, potentially altering its aspect ratio. • Random Cropping: Extracting a random rectangular portion of the resized image, which may include or exclude important features.
# label-aware resizing and cropping: • Label-Aware Resizin

import tensorflow as tf

# Random Resizing and Cropping
def random_resize_crop(image, min_scale, max_scale, target_size):
  # Randomly select a scale factor
  scale = tf.random.uniform([], min_scale, max_scale)

  # Resize the image
  resized_image = tf.image.resize(image, size=[int(image.shape[0] * scale), int(image.shape[1] * scale)])

  # Randomly crop the resized image
  cropped_image = tf.image.random_crop(resized_image, size=[target_size, target_size, 3])

  return cropped_image

# Label-Aware Resizing and Cropping
def label_aware_resize_crop(image, labels, target_size):
  # Calculate bounding box coordinates
  ymin, xmin, ymax, xmax = tf.unstack(labels)
  bbox = tf.stack([ymin, xmin, ymax - ymin, xmax - xmin], axis=0)

  # Resize the image while preserving the bounding box
  resized_image, resized_bbox = tf.image.resize_with_crop_or_pad_box(
      image,
      target_height=target_size,
      target_width=target_size,
      offset_height=int(bbox[0] * target_size),
      offset_width=int(bbox[1] * target_size),
      target_height=int(bbox[2] * target_size),
      target_width=int(bbox[3] * target_size)
  )

  return resized_image, resized_bbox
